The main historical question is wondering if the places on the book_places_wikidata.csv file can be placed onto a map accurately. The file used for map is book_places_wikidata.csv. This file has multiple columns of important info, but primarily the coordinates of the cities are saved in the file.

In [2]:
import pandas as pd
# CODE GENERATED BY GEMINI
# Load the file directly from your Colab environment's current folder
filename = "book_places_wikidata.csv"
df = pd.read_csv(filename)

print(f"\n{'='*50}\nSUCCESSFULLY LOADED: {filename}\n{'='*50}\n")

# 1. Number of rows and columns
num_rows, num_cols = df.shape
print(f"1. Dimensions: {num_rows} rows and {num_cols} columns\n")

# 2. Complete list of column names
print("2. Column Names:")
for col in df.columns:
    print(f"   - {col}")
print()

# 3. First five rows
print("3. First Five Rows:")
display(df.head())
print()

# 4. Number of unique edition_id values
if 'edition_id' in df.columns:
    unique_editions = df['edition_id'].nunique()
    print(f"4. Number of unique edition_id values: {unique_editions}\n")
else:
    print("4. Note: 'edition_id' column was not found in this dataset.\n")

# 5. Missing-value counts for specific columns
coord_candidates = [col for col in df.columns if 'coord' in col.lower() or 'lat' in col.lower() or 'lon' in col.lower()]
coord_col = coord_candidates[0] if coord_candidates else 'coordinate'

check_cols = ['place', 'place_interpreted', 'wikidata_id']
if coord_col in df.columns:
    check_cols.append(coord_col)

print("5. Missing-Value Counts (Number of empty/null entries per column):")
missing_counts = df[check_cols].isnull().sum()
for col, count in missing_counts.items():
    print(f"   - {col}: {count} missing values")
print()

# 6. Distinct sample values from the coordinate column
if coord_col in df.columns:
    print(f"6. Distinct sample values from '{coord_col}' (up to 10 examples):")
    sample_coords = df[coord_col].dropna().unique()[:10]
    for val in sample_coords:
        print(f"   - {val}")
else:
    print(f"6. Could not locate a coordinate column automatically. Detected columns: {df.columns.tolist()}")


SUCCESSFULLY LOADED: book_places_wikidata.csv

1. Dimensions: 500 rows and 19 columns

2. Column Names:
   - edition_id
   - short_title
   - author
   - printer
   - place
   - place_norm
   - place_interpreted
   - country 2
   - coordinate location
   - wikidata_id
   - place_relationship
   - match_status
   - decision_note
   - year
   - language
   - format
   - country
   - year_numeric
   - decade

3. First Five Rows:


,edition_id,short_title,author,printer,place,place_norm,place_interpreted,country 2,coordinate location,wikidata_id,place_relationship,match_status,decision_note,year,language,format,country,year_numeric,decade
0,627908,Das Newe Testament,NaN,Kunigunde Hergot,Nürnberg,Nürnberg,Nuremberg,Germany,"49.45388888888889,11.0775",Q2090,stated,confirmed,NaN,1535,German,8o,Holy Roman Empire,1535,1530s
1,698366,Trewung vom liebebande oder hertztrewen. Item:...,"Streuber, Peter",Zacharias Bärwald,Leipzig,Leipzig,Leipzig,Germany,"51.3406321,12.3747329",Q2079,stated,confirmed,NaN,1590,German,8o,Holy Roman Empire,1590,1590s
2,691468,Romischer Keyserlicher Majestat ordnung und Re...,NaN,Johann Schöffer,Mainz,Mainz,Mainz,NaN,NaN,NaN,stated,ambiguous,NaN,1530,German,2o,Holy Roman Empire,1530,1530s
3,708446,Zwo predigten vom gantzen Christlichen wesen u...,"Spangenberg, Cyriacus",Samuel Emmel,Strasbourg,Strasbourg,Strasbourg,France,"48.573333333333,7.7522222222222",Q6602,stated,confirmed,NaN,1564,German,8o,Holy Roman Empire,1564,1560s
4,640867,"Corpus juris civilis, institutiones dn. Joachi...",NaN,Nikolaus II Episcopius,Basel,Basel,Basel,Switzerland,"47.558055555555555,7.587777777777777",Q78,stated,confirmed,NaN,1555,Latin,2o,Swiss Confederation,1555,1550s



4. Number of unique edition_id values: 500

5. Missing-Value Counts (Number of empty/null entries per column):
   - place: 0 missing values
   - place_interpreted: 0 missing values
   - wikidata_id: 6 missing values
   - coordinate location: 6 missing values

6. Distinct sample values from 'coordinate location' (up to 10 examples):
   - 49.45388888888889,11.0775
   - 51.3406321,12.3747329
   - 48.573333333333,7.7522222222222
   - 47.558055555555555,7.587777777777777
   - 48.36888888888889,10.897777777777778
   - 50.942222222222,6.9577777777778
   - 50.97805555555556,11.028888888888888
   - 52.1315889,11.6399609
   - 50.11055555555556,8.682222222222222
   - 47.37444444444444,8.54111111111111


In [4]:
import pandas as pd
import numpy as np

# Load the already uploaded CSV directly into 'books'
books = pd.read_csv('book_places_wikidata.csv')

# Create our working copy
books_coordinates = books.copy()

# 1. Coordinate format comment:
# Format identified: Comma-separated decimal degrees.
# Source order: Latitude-longitude (e.g., "49.453888,11.0775" represents Nuremberg,
# where 49.45 is the latitude and 11.07 is the longitude).

def parse_coordinate(val):
    """Helper function to parse and validate coordinate strings."""
    if pd.isna(val) or str(val).strip() == '':
        return np.nan, np.nan, 'missing_value'

    try:
        parts = str(val).split(',')
        if len(parts) != 2:
            return np.nan, np.nan, 'format_error'

        lat = float(parts[0].strip())
        lon = float(parts[1].strip())

        # 7. Check ranges: latitude [-90, 90] and longitude [-180, 180]
        if not (-90 <= lat <= 90 and -180 <= lon <= 180):
            return np.nan, np.nan, 'out_of_range'

        return lat, lon, 'success'
    except Exception:
        return np.nan, np.nan, 'format_error'

# 3, 4, 5, 6. Apply parsing, preserve original column, create new numerical and status columns
parsed_results = books_coordinates['coordinate location'].apply(parse_coordinate)
books_coordinates['latitude'] = [r[0] for r in parsed_results]
books_coordinates['longitude'] = [r[1] for r in parsed_results]
books_coordinates['coordinate_parse_status'] = [r[2] for r in parsed_results]

print(f"{'='*60}\nVALIDATION REPORT\n{'='*60}\n")

# 8. Print input/output row counts and assert equality
input_rows = len(books)
output_rows = len(books_coordinates)
print(f"8. Row Count Check:")
print(f"   - Input rows: {input_rows} | Output rows: {output_rows}")
assert input_rows == output_rows, "Row counts do not match!"
print("   -> Assertion passed: Row counts are equal.\n")

# 9. Print unique edition_id count and confirm uniqueness
unique_editions = books_coordinates['edition_id'].nunique()
print(f"9. Edition ID Uniqueness Check:")
print(f"   - Total rows: {output_rows}")
print(f"   - Unique edition_id values: {unique_editions}")
if unique_editions == output_rows:
    print("   -> Confirmation: edition_id remains completely unique.\n")
else:
    print("   -> Note: edition_id has duplicate values.\n")

# 10. Print number and percentage of rows with usable coordinates
usable_count = (books_coordinates['coordinate_parse_status'] == 'success').sum()
usable_pct = (usable_count / output_rows) * 100
print(f"10. Usable Coordinates Summary:")
print(f"    - Usable coordinates: {usable_count} out of {output_rows} ({usable_pct:.2f}%)\n")

# 11. Print every distinct original coordinate value that could not be parsed with its status
print("11. Unparsable Coordinate Values:")
unparsable_df = books_coordinates[books_coordinates['coordinate_parse_status'] != 'success']
if len(unparsable_df) > 0:
    distinct_unparsable = unparsable_df[['coordinate location', 'coordinate_parse_status']].drop_duplicates()
    for idx, row in distinct_unparsable.iterrows():
        print(f"    - Value: '{row['coordinate location']}' | Status: {row['coordinate_parse_status']}")
else:
    print("    - None found (all coordinates successfully parsed).")
print()

# 12. List distinct place_interpreted values without usable coordinates
print("12. Distinct 'place_interpreted' Values Without Usable Coordinates:")
no_usable_places = unparsable_df['place_interpreted'].unique()
if len(no_usable_places) > 0:
    for place in no_usable_places:
        print(f"    - {place}")
else:
    print("    - None.")
print()

# 13. Report any place_interpreted assigned more than one coordinate pair
print("13. Places with Multiple Coordinate Pairs:")
valid_subset = books_coordinates.dropna(subset=['latitude', 'longitude'])
place_coord_counts = valid_subset.groupby('place_interpreted')['coordinate location'].nunique()
multi_coord_places = place_coord_counts[place_coord_counts > 1]
if len(multi_coord_places) > 0:
    print(multi_coord_places)
else:
    print("    - None found (each place has a consistent coordinate pair).")
print()

# 14. Report any coordinate pair assigned to more than one place_interpreted value
print("14. Coordinate Pairs Assigned to Multiple Places:")
coord_place_counts = valid_subset.groupby('coordinate location')['place_interpreted'].nunique()
multi_place_coords = coord_place_counts[coord_place_counts > 1]
if len(multi_place_coords) > 0:
    print(multi_place_coords)
else:
    print("    - None found (each coordinate pair maps to a single place).")
print()

# 15. Print counts by match_status and coordinate_parse_status
print("15. Crosstab of match_status vs. coordinate_parse_status:")
crosstab_result = pd.crosstab(books_coordinates['match_status'], books_coordinates['coordinate_parse_status'], margins=True)
print(crosstab_result)
print()

# 16. Display representative rows
print("16. Representative Rows Sample:")
cols_to_display = ['place', 'place_interpreted', 'match_status', 'coordinate location', 'latitude', 'longitude', 'coordinate_parse_status']
display(books_coordinates[cols_to_display].head())

VALIDATION REPORT

8. Row Count Check:
   - Input rows: 500 | Output rows: 500
   -> Assertion passed: Row counts are equal.

9. Edition ID Uniqueness Check:
   - Total rows: 500
   - Unique edition_id values: 500
   -> Confirmation: edition_id remains completely unique.

10. Usable Coordinates Summary:
    - Usable coordinates: 494 out of 500 (98.80%)

11. Unparsable Coordinate Values:
    - Value: 'nan' | Status: missing_value

12. Distinct 'place_interpreted' Values Without Usable Coordinates:
    - Mainz
    - Haguenau (Augsburg
    - Homberg
    - Basel (Genève

13. Places with Multiple Coordinate Pairs:
    - None found (each place has a consistent coordinate pair).

14. Coordinate Pairs Assigned to Multiple Places:
    - None found (each coordinate pair maps to a single place).

15. Crosstab of match_status vs. coordinate_parse_status:
coordinate_parse_status  missing_value  success  All
match_status                                        
ambiguous                            3 

,place,place_interpreted,match_status,coordinate location,latitude,longitude,coordinate_parse_status
0,Nürnberg,Nuremberg,confirmed,"49.45388888888889,11.0775",49.453889,11.077500,success
1,Leipzig,Leipzig,confirmed,"51.3406321,12.3747329",51.340632,12.374733,success
2,Mainz,Mainz,ambiguous,NaN,NaN,NaN,missing_value
3,Strasbourg,Strasbourg,confirmed,"48.573333333333,7.7522222222222",48.573333,7.752222,success
4,Basel,Basel,confirmed,"47.558055555555555,7.587777777777777",47.558056,7.587778,success


In [5]:
import pandas as pd

# Ensure we are working with books_coordinates from the previous step
# (books_coordinates and books are preserved untouched)

# 2. Use only records with usable numerical latitude and longitude
valid_books = books_coordinates[books_coordinates['coordinate_parse_status'] == 'success'].copy()

# 3. Report excluded records (missing/invalid coordinates) and list their details
invalid_books = books_coordinates[books_coordinates['coordinate_parse_status'] != 'success']
print(f"{'='*60}\nEXCLUSION REPORT FOR MAPPING\n{'='*60}")
print(f"Number of records excluded: {len(invalid_books)}")
print("Distinct place_interpreted and match_status of excluded records:")
display(invalid_books[['place_interpreted', 'match_status', 'coordinate_parse_status']].drop_duplicates())
print()

# 4. Count edition_id values and confirm uniqueness in the original dataframe
total_orig_rows = len(books_coordinates)
unique_editions = books_coordinates['edition_id'].nunique()
print(f"4. Edition ID Uniqueness Check:")
print(f"   - Total rows in original dataframe: {total_orig_rows}")
print(f"   - Unique edition_id count: {unique_editions}")
assert unique_editions == total_orig_rows, "Assertion Error: edition_id is not unique!"
print("   -> Confirmation: edition_id is completely unique in the original dataframe.\n")

# 5. Check whether one place_interpreted value has been assigned more than one coordinate pair
place_coord_counts = valid_books.groupby('place_interpreted')['coordinate location'].nunique()
multi_coord_places = place_coord_counts[place_coord_counts > 1]
print(f"5. Multi-Coordinate Place Check:")
if len(multi_coord_places) > 0:
    print("   -> Warning: Found place(s) with multiple coordinate pairs:")
    print(multi_coord_places)
else:
    print("   -> Passed: Each place_interpreted has a single, consistent coordinate pair.\n")

# Create city_counts: one row for each unique combination of place_interpreted, latitude, longitude
city_counts = valid_books.groupby(
    ['place_interpreted', 'latitude', 'longitude']
).size().reset_index(name='editions')

# 6. Sort city_counts from largest to smallest edition count
city_counts = city_counts.sort_values(by='editions', ascending=False).reset_index(drop=True)

print("6. Top 10 Rows of 'city_counts' (Sorted by Editions):")
display(city_counts.head(10))
print()

# 7. Print number of usable rows and sum of editions, then assert equality
usable_rows_count = len(valid_books)
sum_editions = city_counts['editions'].sum()
print(f"7. Aggregation Total Check:")
print(f"   - Number of rows with usable coordinates: {usable_rows_count}")
print(f"   - Sum of the 'editions' column in city_counts: {sum_editions}")
assert usable_rows_count == sum_editions, "Assertion Error: Totals do not match!"
print("   -> Assertion passed: Usable row count equals the sum of aggregated editions.\n")

# 8. Save city_counts as CSV without a pandas index column
output_filename = "printing_places_aggregated.csv"
city_counts.to_csv(output_filename, index=False)
print(f"8. Successfully saved aggregated table as '{output_filename}' (without index).\n")

EXCLUSION REPORT FOR MAPPING
Number of records excluded: 6
Distinct place_interpreted and match_status of excluded records:


,place_interpreted,match_status,coordinate_parse_status
2,Mainz,ambiguous,missing_value
92,Haguenau (Augsburg,unmatched,missing_value
174,Homberg,ambiguous,missing_value
204,Basel (Genève,unmatched,missing_value



4. Edition ID Uniqueness Check:
   - Total rows in original dataframe: 500
   - Unique edition_id count: 500
   -> Confirmation: edition_id is completely unique in the original dataframe.

5. Multi-Coordinate Place Check:
   -> Passed: Each place_interpreted has a single, consistent coordinate pair.

6. Top 10 Rows of 'city_counts' (Sorted by Editions):


,place_interpreted,latitude,longitude,editions
0,Basel,47.558056,7.587778,135
1,Nuremberg,49.453889,11.077500,69
2,Cologne,50.942222,6.957778,69
3,Strasbourg,48.573333,7.752222,56
4,Leipzig,51.340632,12.374733,49
5,Augsburg,48.368889,10.897778,37
6,Zurich,47.374444,8.541111,18
7,Erfurt,50.978056,11.028889,15
8,Tübingen,48.520000,9.055556,14
9,Magdeburg,52.131589,11.639961,11



7. Aggregation Total Check:
   - Number of rows with usable coordinates: 494
   - Sum of the 'editions' column in city_counts: 494
   -> Assertion passed: Usable row count equals the sum of aggregated editions.

8. Successfully saved aggregated table as 'printing_places_aggregated.csv' (without index).



In [6]:
!pip install folium

In [7]:
import folium
import numpy as np
import pandas as pd

# Ensure city_counts is loaded (or use the existing DataFrame)
# city_counts = pd.read_csv("printing_places_aggregated.csv")

# 1 & 2. Create base map centered on Central Europe with no default tiles,
# then add the custom OpenFreeMap Natural Earth tile layer
m = folium.Map(location=[50.0, 10.0], zoom_start=4, tiles=None)

folium.TileLayer(
    tiles='https://tiles.openfreemap.org/natural_earth/ne2sr/{z}/{x}/{y}.png',
    attr='&copy; <a href="https://openfreemap.org">OpenFreeMap</a> &copy; <a href="https://www.naturalearthdata.com">Natural Earth</a>',
    name='OpenFreeMap Natural Earth',
    max_zoom=6,
    min_zoom=0
).add_to(m)

# 3, 4, 5, 6, 7. Iterate through city_counts and create CircleMarkers
for idx, row in city_counts.iterrows():
    place = row['place_interpreted']
    lat = row['latitude']
    lon = row['longitude']
    editions = row['editions']

    # 4. Set radius so area is approximately proportional to the edition count
    radius = 3 + 2.2 * np.sqrt(editions)

    # 6. Tooltip with place name and edition count
    tooltip_text = f"{place}: {editions} editions"

    # 7. Popup with place, count, latitude, and longitude
    popup_text = (
        f"<b>Place:</b> {place}<br>"
        f"<b>Editions:</b> {editions}<br>"
        f"<b>Latitude:</b> {lat}<br>"
        f"<b>Longitude:</b> {lon}"
    )

    # 5. Partly transparent fill and visible border
    folium.CircleMarker(
        location=[lat, lon],
        radius=radius,
        color='#1f78b4',         # Visible border color
        weight=1.5,
        fill=True,
        fill_color='#33a02c',    # Partly transparent fill color
        fill_opacity=0.6,
        tooltip=tooltip_text,
        popup=popup_text
    ).add_to(m)

# 8. Fit the final view to the plotted points if possible
coordinates_list = city_counts[['latitude', 'longitude']].values.tolist()
if len(coordinates_list) > 0:
    m.fit_bounds(coordinates_list)

# 9. Save the result as "printing_places_map.html"
output_filename = "printing_places_map.html"
m.save(output_filename)
print(f"Map successfully saved as '{output_filename}'.")

# 10. Display the map directly inside the Google Colab notebook
display(m)

Map successfully saved as 'printing_places_map.html'.
